# Montra Hospitality Intel: Topic Modelling with BERTopic
**Guest Experience Analytics & Topic Modelling**

This notebook applies BERTopic to discover latent, operational hospitality themes from unstructured guest reviews.
Geographic entity stripping is performed (removing UK city names so topics cluster around service quality rather than location), documents are embedded using Sentence Transformers (`all-MiniLM-L6-v2`), clustered via UMAP + HDBSCAN, topic representations extracted using class-based TF-IDF (c-TF-IDF), topics reduced, and outlier feedback reassigned.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earlchirchir/Guest-Experience-Analytics/blob/main/notebook/03_topic_modelling.ipynb)

> **Note**: Run this notebook on Google Colab with GPU hardware acceleration (Runtime > Change runtime type > T4 GPU) for fast transformer embedding generation and BERTopic clustering.

In [ ]:
# Install dependencies in Google Colab
!pip install -q bertopic sentence-transformers umap-learn hdbscan


In [ ]:
import os
import re
from collections import Counter
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

# Load preprocessed guest feedback data (local workspace or Colab)
input_path = "../data/processed/montra_cleaned_data.csv"
if not os.path.exists(input_path):
    if os.path.exists("data/processed/montra_cleaned_data.csv"):
        input_path = "data/processed/montra_cleaned_data.csv"
    elif os.path.exists("montra_cleaned_data.csv"):
        input_path = "montra_cleaned_data.csv"
    else:
        input_path = "https://raw.githubusercontent.com/earlchirchir/Guest-Experience-Analytics/main/data/processed/montra_cleaned_data.csv"

merged = pd.read_csv(input_path)
print("Loaded preprocessed data shape:", merged.shape)
merged.head()


In [ ]:
# Extract review document list
document = merged['cleaned_review_text'].astype(str).tolist()
print(f"Total review documents: {len(document)}")
print("Sample documents:")
for i, doc in enumerate(document[:3]):
    print(f"[{i+1}] {doc}")


### 1. Remove Geographic Noise (UK Cities)
To prevent the topic model from clustering reviews by geographic location (e.g., separating London hotels from Edinburgh hotels), all UK city names are stripped out so the clusters capture operational hospitality themes.

In [ ]:
# UK Cities list to eliminate geographic bias
UK_Cities = [
    "London", "Manchester", "Edinburgh", "Birmingham", "Leeds",
    "Bristol", "Liverpool", "Glasgow", "Oxford", "Bath",
    "York", "Cardiff", "New Castle", "Nottingham", "Brighton",
    "Sheffield", "Leicester", "Belfast", "Southampton", "Aberdeen"
]

# Compile case-insensitive regex pattern and remove city references
city_pattern = re.compile(r'\b(' + '|'.join(UK_Cities) + r')\b', re.IGNORECASE)
document = [city_pattern.sub('', doc).strip() for doc in document]
document = [re.sub(r'\s+', ' ', doc) for doc in document]

print("Sanitized documents preview:")
for i, doc in enumerate(document[:3]):
    print(f"[{i+1}] {doc}")


### 2. Vocabulary & Duplicate Review Inspection
Document collection vocabulary analysis and inspection of duplicated normalized reviews.

In [ ]:
# Count exact duplicate normalized reviews
normalized_docs = [doc.lower().strip() for doc in document]
total_docs = len(normalized_docs)
unique_docs = len(set(normalized_docs))
duplicated_docs = total_docs - unique_docs

print(f"Total documents: {total_docs}")
print(f"Unique documents: {unique_docs}")
print(f"Duplicate documents: {duplicated_docs}")

# Check vocabulary size and top 20 most frequent words
all_words = " ".join(document).split()
print("Vocab size:", len(set(all_words)))
print("\nTop 20 most common tokens:")
print(Counter(all_words).most_common(20))


### 3. Generate Semantic Sentence Embeddings
The pretrained Sentence Transformer model `all-MiniLM-L6-v2` converts each review into a dense 384-dimensional semantic embedding vector.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load sentence transformer model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings (runs with progress bar)
embeddings = embedding_model.encode(document, show_progress_bar=True)
print("Embeddings array shape:", embeddings.shape)


### 4. Build and Train BERTopic Model
Modular BERTopic architecture configuration:
1. **UMAP**: Reduces dense 384-d vectors into 5 dimensions for clustering while preserving local/global manifold structure.
2. **HDBSCAN**: Identifies dense clusters of variable shape without forcing noise into clusters.
3. **CountVectorizer**: Generates candidate n-grams (1 to 3 words) ignoring standard English stopwords.
4. **c-TF-IDF**: Scores how distinctive words are for each cluster relative to the overall corpus.

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

# 1. Dimensionality reduction with UMAP
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# 2. Density-based clustering with HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# 3. Tokenizer & Stopwords vectorizer
vectorizer_model = CountVectorizer(
    min_df=5,
    ngram_range=(1, 3),
    stop_words="english"
)

# 4. Class-based TF-IDF
ctfidf_model = ClassTfidfTransformer()

# 5. Assemble BERTopic pipeline
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=True,
    verbose=True
)

# Fit and transform
topics, probabilities = topic_model.fit_transform(document, embeddings)
print("Initial topic extraction completed!")


In [ ]:
# Display extracted topics overview
topic_info = topic_model.get_topic_info()
print(f"Total initial topics discovered: {len(topic_info)}")
display(topic_info.head(15))


### 5. Topic Reduction & Outlier Reassignment
Topic reduction and outlier reassignment:
1. Redundant micro-topics are reduced using `reduce_topics(document, nr_topics='auto')` or by setting a target number of operational hospitality clusters (e.g., 7 major pillars: Staff, Cleanliness, Comfort, Dining, Amenities, Location, Value).
2. Outlier reviews (topic -1) are reassigned to their closest semantic topic based on probabilities.

In [ ]:
# Reduce topics automatically to consolidate granular sub-themes
topic_model.reduce_topics(document, nr_topics="auto")

# Reassign outlier documents (topic -1) to their highest probability topic
new_topics = topic_model.reduce_outliers(document, topics, strategy="c-tf-idf")
topic_model.update_topics(document, topics=new_topics)

print("Updated Topic Information after reduction and outlier handling:")
topic_info_reduced = topic_model.get_topic_info()
display(topic_info_reduced.head(10))


### 6. Topic Visualizations
Visual diagnostics to explore topic semantics and intertopic relationships:
- Topic word score barcharts
- Intertopic distance maps
- Topic hierarchy dendrogram

In [ ]:
# Top terms per topic barchart
fig_barchart = topic_model.visualize_barchart(top_n_topics=8, n_words=8)
fig_barchart.show()


In [ ]:
# Intertopic distance map
fig_topics = topic_model.visualize_topics()
fig_topics.show()


In [ ]:
# Topic hierarchy dendrogram
fig_hierarchy = topic_model.visualize_hierarchy()
fig_hierarchy.show()


### 7. Export Topic Modelling Results
Assigned topic IDs and representative keywords are attached back to the original dataset and saved to `../data/processed/reviews_with_topics.csv`.

In [ ]:
# Map topic labels
topic_labels = {
    row['Topic']: row['Name'] for _, row in topic_model.get_topic_info().iterrows()
}

merged['topic_id'] = new_topics
merged['topic_name'] = merged['topic_id'].map(topic_labels)

# Save enriched dataset
output_path = "../data/processed/reviews_with_topics.csv"
merged.to_csv(output_path, index=False)
print(f"Successfully saved {len(merged)} reviews with topic assignments to {output_path}")

# Save trained topic model
os.makedirs("../model", exist_ok=True)
topic_model.save("../model/bertopic_model", serialization="safetensors", save_embedding_model=False)
print("Successfully saved trained topic model to ../model/bertopic_model")

# Display topic distribution
print("\nReview counts per topic:")
print(merged['topic_name'].value_counts())
